# 🐭 Remy.IA · Coleta de receitas do TudoGostoso

Este notebook monta a **base de receitas brasileiras** do Remy.IA a partir do site [TudoGostoso](https://www.tudogostoso.com.br).

**Uso:** acadêmico, com autorização da Webedia, dona do site. Os textos e as notas são deles.
- **Não publique a base coletada** (CSV, JSON) em repositório público, como o GitHub. Guarde só no Drive do grupo. A `.gitignore` do projeto já ignora a pasta de dados.
- Cite o TudoGostoso como fonte no relatório e nos slides.

## Como o scraper trabalha
1. Lê o **índice público do site** (`sitemap.xml`), que lista o link de todas as receitas.
2. Visita **uma receita por vez**, com **pausa entre as visitas**, para não sobrecarregar o site.
3. Respeita o `robots.txt` e para sozinho se o site pedir para ir mais devagar (erro 429).
4. Salva cada receita assim que lê, no **seu Google Drive**. Se o Colab cair, rode de novo: ele **continua de onde parou**.

## Quanto tempo leva
O site tem dezenas de milhares de receitas. Com 1 segundo de pausa, são **~3.600 receitas por hora**.
- O Colab grátis desliga depois de algumas horas. Por isso a coleta é **em partes**: rode, deixe trabalhando, e quando cair, rode de novo.
- Para testar, comece com `MAX_RECEITAS = 200`.

## Passo a passo
1. **Ambiente de execução › Executar tudo**. Não precisa de GPU.
2. Autorize o acesso ao Google Drive quando pedir.
3. Confira o **teste com 5 receitas** (seção 4). Se os campos vierem vazios, pare e avise: o site pode ter mudado.
4. Deixe a coleta rodando (seção 5).
5. A seção 6 gera a base final: `receitas_tudogostoso.csv` e `.parquet` na pasta `RemyIA_dados` do seu Drive.

## 1. Configuração
Rode e autorize o Google Drive. Ajuste `PAUSA` e `MAX_RECEITAS` se quiser.

In [ ]:
# ============ 1. CONFIGURAÇÃO ============
!pip install -q requests beautifulsoup4 lxml tqdm pyarrow

import os, re, json, time, random, html, unicodedata
from datetime import datetime, timezone
from urllib import robotparser
import xml.etree.ElementTree as ET
import requests
import pandas as pd
from bs4 import BeautifulSoup
from tqdm.auto import tqdm

SITE = "https://www.tudogostoso.com.br"
PAUSA = 1.0               # segundos entre uma receita e outra (não diminua muito: respeite o site)
MAX_RECEITAS = None       # None = todas. Para testar, use 200.
SALVAR_NO_DRIVE = True    # False = salva só na sessão do Colab (some quando ela fecha)

# Identifique o projeto. Troque o contato pelo e-mail do grupo, se quiser que o site possa falar com vocês.
USER_AGENT = "RemyIA-Uniso/1.0 (projeto academico; coleta autorizada; contato: COLOQUE_O_EMAIL_DO_GRUPO)"

if SALVAR_NO_DRIVE:
    try:
        from google.colab import drive
        drive.mount("/content/drive")
        PASTA = "/content/drive/MyDrive/RemyIA_dados"
    except ImportError:
        PASTA = "RemyIA_dados"
else:
    PASTA = "/content/RemyIA_dados"
os.makedirs(PASTA, exist_ok=True)

ARQ_URLS = f"{PASTA}/urls_receitas.csv"
ARQ_BRUTO = f"{PASTA}/receitas_bruto.jsonl"       # uma receita por linha, gravada na hora
ARQ_ERROS = f"{PASTA}/erros.jsonl"
print("Os dados vão para:", PASTA)

sessao = requests.Session()
sessao.headers.update({"User-Agent": USER_AGENT, "Accept-Language": "pt-BR,pt;q=0.9"})

robos = robotparser.RobotFileParser(f"{SITE}/robots.txt")
robos.read()
print("robots.txt permite /receita/?", robos.can_fetch(USER_AGENT, f"{SITE}/receita/23-bolo-de-cenoura.html"))

def baixar(url, tentativas=4):
    """GET educado: respeita robots.txt, espera mais quando o site pede (429/503) e tenta de novo."""
    if not robos.can_fetch(USER_AGENT, url):
        raise PermissionError(f"robots.txt não permite: {url}")
    espera = 5
    for t in range(tentativas):
        try:
            r = sessao.get(url, timeout=30)
        except requests.RequestException as e:
            if t == tentativas - 1: raise
            time.sleep(espera); espera *= 2; continue
        if r.status_code == 200:
            return r.text
        if r.status_code in (429, 503):
            pausa = int(r.headers.get("Retry-After", espera))
            print(f"⏳ O site pediu para ir mais devagar. Esperando {pausa}s...")
            time.sleep(pausa); espera *= 2; continue
        if r.status_code == 404:
            return None
        if t == tentativas - 1:
            r.raise_for_status()
        time.sleep(espera); espera *= 2
    return None

## 2. Lista de receitas
Lê o índice público do site e salva todos os links de receitas. Só roda uma vez.

In [ ]:
# ============ 2. LISTA DE RECEITAS (pelo sitemap) ============
# O sitemap.xml é um índice com vários arquivos menores; cada um lista links de receitas.
NS = {"s": "http://www.sitemaps.org/schemas/sitemap/0.9"}
PADRAO_RECEITA = re.compile(r"/receita/(\d+)-[^/]+\.html$")

def ler_sitemap(url):
    texto = baixar(url)
    if not texto: return [], []
    raiz = ET.fromstring(texto.encode("utf-8"))
    filhos = [e.text.strip() for e in raiz.findall("s:sitemap/s:loc", NS)]
    links = [e.text.strip() for e in raiz.findall("s:url/s:loc", NS)]
    return filhos, links

if os.path.exists(ARQ_URLS):
    urls = pd.read_csv(ARQ_URLS)
    print(f"Lista já existe: {len(urls):,} receitas (apague {ARQ_URLS} para refazer)")
else:
    filhos, links = ler_sitemap(f"{SITE}/sitemap.xml")
    for f in tqdm(filhos, desc="Lendo o sitemap"):
        _, l = ler_sitemap(f)
        links += l
        time.sleep(PAUSA)
    linhas = [{"id": int(m.group(1)), "url": u} for u in links if (m := PADRAO_RECEITA.search(u))]
    urls = pd.DataFrame(linhas).drop_duplicates("id").sort_values("id").reset_index(drop=True)
    urls.to_csv(ARQ_URLS, index=False)
    print(f"{len(urls):,} receitas encontradas no sitemap")

urls.head()

## 3. Como uma receita é lida
Nome, categoria, tempo, porções, dificuldade, **nota e número de avaliações**, ingredientes (com as seções, como *Massa* e *Cobertura*) e modo de preparo.

In [ ]:
# ============ 3. LEITURA DE UMA RECEITA ============
# Primeiro tenta os dados estruturados (JSON-LD, padrão schema.org que sites de receita usam para o Google).
# Se não tiver, lê o HTML da página: título, ingredientes por seção, modo de preparo, tempo, porções e nota.

def _txt(el):
    return re.sub(r"\s+", " ", el.get_text(" ", strip=True)).strip() if el else ""

def _limpa(t):
    """Desfaz códigos HTML que vêm no JSON-LD ('ficar&aacute;' -> 'ficará')."""
    return re.sub(r"\s+", " ", html.unescape(html.unescape(str(t)))).strip()

def _minutos_iso(v):
    """'PT1H20M' -> 80"""
    if not v or not isinstance(v, str): return None
    m = re.match(r"P(?:T)?(?:(\d+)H)?(?:(\d+)M)?", v.strip())
    if not m or not any(m.groups()): return None
    return int(m.group(1) or 0) * 60 + int(m.group(2) or 0)

def _minutos_texto(t):
    """'1h 20min', '40 min', '1 hora' -> minutos"""
    h = re.search(r"(\d+)\s*(?:h|hora)", t, re.I)
    m = re.search(r"(\d+)\s*min", t, re.I)
    if not h and not m: return None
    return (int(h.group(1)) * 60 if h else 0) + (int(m.group(1)) if m else 0)

def _numero(v):
    try: return float(str(v).replace(",", "."))
    except (TypeError, ValueError): return None

def _recipe_jsonld(sopa):
    for s in sopa.find_all("script", type="application/ld+json"):
        try: dados = json.loads(s.string or s.get_text() or "")
        except json.JSONDecodeError: continue
        fila = dados if isinstance(dados, list) else [dados]
        while fila:
            d = fila.pop(0)
            if not isinstance(d, dict): continue
            fila += d.get("@graph", []) if isinstance(d.get("@graph"), list) else []
            tipo = d.get("@type")
            if tipo == "Recipe" or (isinstance(tipo, list) and "Recipe" in tipo):
                return d
    return None

def _passos_jsonld(instr):
    out = []
    def visita(x, secao=None):
        if isinstance(x, str):
            for p in re.split(r"\n+", x):
                if p.strip(): out.append({"secao": secao, "texto": p.strip()})
        elif isinstance(x, list):
            for y in x: visita(y, secao)
        elif isinstance(x, dict):
            if x.get("@type") == "HowToSection":
                visita(x.get("itemListElement", []), x.get("name"))
            else:
                visita(x.get("text") or x.get("name") or "", secao)
    visita(instr)
    return out

def _lista_por_secao(sopa, titulo_regex):
    """Acha o bloco que começa num título (ex.: 'Ingredientes') e lê os itens em ordem, guardando a seção."""
    titulo = None
    for h in sopa.find_all(re.compile(r"^h[1-4]$")):
        if re.search(titulo_regex, _txt(h), re.I):
            titulo = h; break
    if titulo is None: return []
    bloco = titulo.parent
    for _ in range(3):                                   # sobe até achar um bloco com itens de lista
        if bloco.find("li"): break
        bloco = bloco.parent
    itens, secao, vistos = [], None, set()
    for el in bloco.find_all(["h2", "h3", "h4", "h5", "strong", "li"]):
        if el is titulo: continue
        if el.name == "li":
            t = _txt(el)
            if t and id(el) not in vistos and not el.find("li"):
                itens.append({"secao": secao, "texto": t}); vistos.add(id(el))
        else:
            t = _txt(el)
            if el.name in ("h2",) and itens: break      # chegou na próxima parte da página
            if 0 < len(t) < 40 and not el.find_parent("li"):
                secao = t.rstrip(":")
    return itens

def ler_receita(url, html_texto):
    sopa = BeautifulSoup(html_texto, "lxml")
    m = PADRAO_RECEITA.search(url)
    r = {"id": int(m.group(1)) if m else None, "url": url, "fonte_dados": None,
         "nome": None, "categoria": None, "tempo_min": None, "porcoes": None, "dificuldade": None,
         "nota": None, "n_avaliacoes": None, "autor": None, "calorias": None, "imagem": None,
         "ingredientes": [], "passos": [], "coletado_em": datetime.now(timezone.utc).isoformat(timespec="seconds")}

    ld = _recipe_jsonld(sopa)
    if ld:
        r["fonte_dados"] = "json-ld"
        r["nome"] = _limpa(ld.get("name")) if ld.get("name") else None
        r["ingredientes"] = [{"secao": None, "texto": _limpa(i)} for i in ld.get("recipeIngredient", []) if i]
        r["passos"] = [dict(p, texto=_limpa(p["texto"])) for p in _passos_jsonld(ld.get("recipeInstructions", []))]
        r["tempo_min"] = _minutos_iso(ld.get("totalTime")) or _minutos_iso(ld.get("cookTime"))
        rend = ld.get("recipeYield")
        rend = rend[0] if isinstance(rend, list) and rend else rend
        if rend: r["porcoes"] = _numero(re.search(r"\d+", str(rend)).group()) if re.search(r"\d+", str(rend)) else None
        av = ld.get("aggregateRating") or {}
        r["nota"] = _numero(av.get("ratingValue"))
        r["n_avaliacoes"] = _numero(av.get("ratingCount") or av.get("reviewCount"))
        cat = ld.get("recipeCategory")
        r["categoria"] = cat[0] if isinstance(cat, list) and cat else cat
        aut = ld.get("author")
        aut = aut[0] if isinstance(aut, list) and aut else aut
        r["autor"] = aut.get("name") if isinstance(aut, dict) else aut
        nut = ld.get("nutrition") or {}
        r["calorias"] = _numero(re.sub(r"[^\d.,]", "", str(nut.get("calories", "")))) if nut.get("calories") else None
        img = ld.get("image")
        img = img[0] if isinstance(img, list) and img else img
        r["imagem"] = img.get("url") if isinstance(img, dict) else img

    # Completa pelo HTML o que faltou (ou tudo, se não houver JSON-LD)
    texto_pagina = _txt(sopa.body) if sopa.body else ""
    if not r["nome"]:
        r["nome"] = _txt(sopa.find("h1")) or None
    ing_html = _lista_por_secao(sopa, r"^ingredientes")
    if ing_html and (not r["ingredientes"] or any(i["secao"] for i in ing_html)):
        r["ingredientes"] = ing_html                      # o HTML guarda as seções (Massa, Cobertura...)
    if not r["passos"]:
        r["passos"] = _lista_por_secao(sopa, r"modo de preparo|preparo")
    if r["fonte_dados"] is None and (r["ingredientes"] or r["passos"]):
        r["fonte_dados"] = "html"
    topo = texto_pagina[:3000]                           # tempo, porções, dificuldade e nota ficam no topo
    if r["tempo_min"] is None: r["tempo_min"] = _minutos_texto(topo)
    if r["porcoes"] is None:
        p = re.search(r"(\d+)\s*porç", texto_pagina, re.I)
        r["porcoes"] = int(p.group(1)) if p else None
    d = re.search(r"\b(Fácil|Médio|Médi[oa]|Difícil)\b", topo)
    r["dificuldade"] = d.group(1) if d else None
    if r["nota"] is None:
        el = sopa.find(attrs={"itemprop": "ratingValue"})
        r["nota"] = _numero(el.get("content") or _txt(el)) if el else None
        if r["nota"] is None:
            n = re.search(r"\b([0-5][.,]\d)\b\s*(?:\(|·|-)?\s*([\d.]+)\s*avalia", topo, re.I)
            if n: r["nota"], r["n_avaliacoes"] = _numero(n.group(1)), _numero(n.group(2).replace(".", ""))
    if r["n_avaliacoes"] is None:
        n = re.search(r"([\d.]+)\s*avalia", topo, re.I)
        r["n_avaliacoes"] = _numero(n.group(1).replace(".", "")) if n else None
    if not r["categoria"]:
        trilha = sopa.find(attrs={"class": re.compile("breadcrumb", re.I)}) or sopa.find("nav", attrs={"aria-label": re.compile("breadcrumb|trilha", re.I)})
        partes = [_txt(a) for a in trilha.find_all(["a", "span", "li"]) if _txt(a)] if trilha else []
        partes = [p for p in partes if p.lower() not in ("home", "início", "tudogostoso", "receitas")]
        r["categoria"] = partes[-1] if partes else None
    if not r["autor"]:
        a = re.search(r"\bPor\s+([A-ZÀ-Ú][\wÀ-ú.]*(?:\s+[A-ZÀ-Ú][\wÀ-ú.]*){0,3})", topo)
        nome = a.group(1).split() if a else []
        while nome and nome[-1] in ("Ingredientes", "Modo", "Tempo", "Rendimento", "Dificuldade", "Custo", "Receita"):
            nome.pop()
        r["autor"] = " ".join(nome) or None
    if not r["imagem"]:
        og = sopa.find("meta", property="og:image")
        r["imagem"] = og.get("content") if og else None
    return r

## 4. Teste
Confira as 5 primeiras antes de deixar a coleta rodando.

In [ ]:
# ============ 4. TESTE COM 5 RECEITAS ============
# Confira se os campos vieram preenchidos. Se nome, ingredientes ou passos vierem vazios, PARE e avise:
# o site pode ter mudado o formato das páginas.
for url in urls["url"].head(5):
    r = ler_receita(url, baixar(url) or "")
    print(f"\n#{r['id']} {r['nome']}  ({r['fonte_dados']})")
    print(f"   categoria: {r['categoria']} · {r['tempo_min']} min · {r['porcoes']} porções · {r['dificuldade']}")
    print(f"   nota: {r['nota']} ({r['n_avaliacoes']} avaliações) · autor: {r['autor']}")
    print(f"   {len(r['ingredientes'])} ingredientes, ex.: {[i['texto'] for i in r['ingredientes'][:3]]}")
    print(f"   {len(r['passos'])} passos · seções: {sorted({i['secao'] for i in r['ingredientes'] if i['secao']})}")
    time.sleep(PAUSA)

## 5. Coleta
Pode demorar horas. Se o Colab desligar, rode **Executar tudo** de novo: as receitas já salvas são puladas.

In [ ]:
# ============ 5. COLETA (pode rodar várias vezes: continua de onde parou) ============
def ids_ja_coletados():
    feitos = set()
    for arq in (ARQ_BRUTO, ARQ_ERROS):
        if os.path.exists(arq):
            with open(arq, encoding="utf-8") as f:
                for linha in f:
                    try: feitos.add(json.loads(linha)["id"])
                    except (json.JSONDecodeError, KeyError): pass
    return feitos

feitos = ids_ja_coletados()
fila = urls[~urls["id"].isin(feitos)]
if MAX_RECEITAS:
    fila = fila.head(max(0, MAX_RECEITAS - len(feitos)))
print(f"Já coletadas: {len(feitos):,} · faltam nesta rodada: {len(fila):,}")
print(f"Tempo estimado: ~{len(fila) * (PAUSA + 0.4) / 3600:.1f} h")

ok = erros = 0
with open(ARQ_BRUTO, "a", encoding="utf-8") as saida, open(ARQ_ERROS, "a", encoding="utf-8") as log:
    for _, linha in tqdm(fila.iterrows(), total=len(fila), desc="Coletando"):
        try:
            pagina = baixar(linha["url"])
            if not pagina:
                raise ValueError("página não encontrada (404)")
            r = ler_receita(linha["url"], pagina)
            if not r["nome"] or not r["ingredientes"]:
                raise ValueError("página sem nome ou ingredientes")
            saida.write(json.dumps(r, ensure_ascii=False) + "\n"); saida.flush()
            ok += 1
        except PermissionError as e:
            print("⛔", e); break
        except Exception as e:
            log.write(json.dumps({"id": int(linha["id"]), "url": linha["url"], "erro": f"{type(e).__name__}: {e}"},
                                 ensure_ascii=False) + "\n"); log.flush()
            erros += 1
        time.sleep(PAUSA + random.uniform(0, 0.3))

print(f"\nNesta rodada: {ok:,} receitas salvas, {erros:,} com erro (ver {ARQ_ERROS}).")

## 6. Base final
Gera `receitas_tudogostoso.parquet` (para o notebook do Remy.IA) e `.csv` (para abrir no Excel). Pode rodar a qualquer momento, mesmo com a coleta pela metade.

In [ ]:
# ============ 6. BASE FINAL ============
# Junta tudo, tira repetidas e cria a coluna "ingredientes_nomes": só o nome de cada ingrediente,
# sem quantidade nem unidade ("2 xícaras (chá) de farinha de trigo" -> "farinha de trigo").
# É essa coluna que o Remy.IA usa na busca.

def sem_acento(t):
    return "".join(c for c in unicodedata.normalize("NFD", t) if unicodedata.category(c) != "Mn")

UNIDADES = (r"x[ií]caras?|colher(?:es)?|colherinhas?|copos?|latas?|caixas?|caixinhas?|pacotes?|potes?|vidros?|"
            r"envelopes?|sach[eê]s?|tabletes?|barras?|fatias?|dentes?|ramos?|ma[cç]os?|folhas?|pitadas?|punhados?|"
            r"pedacos?|peda[cç]os?|unidades?|un|gr?|gramas?|kg|quilos?|mg|ml|l|litros?|cm|por[cç](?:[aã]o|[oõ]es)|"
            r"talos?|cabe[cç]as?|gotas?|doses?|conchas?|bandejas?|saquinhos?|fios?|cubos?|lascas?")
TAMANHOS = (r"\b(grandes?|m[eé]dios?|m[eé]dias?|pequen[oa]s?|picad[oa]s?|ralad[oa]s?|cozid[oa]s?|fatiad[oa]s?|"
            r"cortad[oa]s?|amassad[oa]s?|derretid[oa]s?|amolecid[oa]s?|peneirad[oa]s?|bem|maduras?|maduros?|"
            r"frescos?|frescas?|gelad[oa]s?|morn[oa]s?|em temperatura ambiente|sem casca|com casca|sem pele|"
            r"descascad[oa]s?|lavad[oa]s?|escorrid[oa]s?|batid[oa]s?|inteir[oa]s?|cheia|cheio|rasa|raso|"
            r"aproximadamente|cerca de|opcional|a gosto|q\.?b\.?|para untar|para polvilhar|para decorar)\b")

def limpar_ingrediente(t):
    t = sem_acento(str(t).lower())
    t = re.sub(r"\(.*?\)", " ", t)                           # "(chá)", "(opcional)"
    t = t.split(",")[0]                                      # ", picada em cubos"
    t = re.sub(r"^\s*(?:[\d½¼¾⅓⅔/.,\-]+|meia|meio|uma?|duas|dois|tres|quatro|cinco|seis|alguns|algumas)\s*", " ", t)
    t = re.sub(r"^\s*(?:e\s+)?[\d½¼¾/]*\s*", " ", t)
    t = re.sub(rf"^\s*(?:{UNIDADES})\b\.?\s*(?:de\s+(?:sopa|cha|cafe|sobremesa)\b)?\s*", " ", t)
    t = re.sub(r"^\s*(?:de|do|da|dos|das)\s+", " ", t)
    t = re.sub(TAMANHOS, " ", t)
    t = re.sub(r"\bem (cubos|cubinhos|tiras|rodelas|pedacos|fatias|bifes|po fino)\b", " ", t) if "fermento em po" not in t else t
    t = re.sub(r"\b(e|ou|para|a)\s*$", " ", t.strip())
    return re.sub(r"\s+", " ", t).strip(" .;:-")

linhas = []
with open(ARQ_BRUTO, encoding="utf-8") as f:
    for linha in f:
        try: linhas.append(json.loads(linha))
        except json.JSONDecodeError: pass
base = pd.DataFrame(linhas).drop_duplicates("id", keep="last").reset_index(drop=True)
base["ingredientes_texto"] = base["ingredientes"].apply(lambda l: [i["texto"] for i in l])
base["ingredientes_nomes"] = base["ingredientes_texto"].apply(lambda l: [n for n in map(limpar_ingrediente, l) if n])
base["passos_texto"] = base["passos"].apply(lambda l: [p["texto"] for p in l])
base["n_ingredientes"] = base["ingredientes_nomes"].str.len()

base.to_parquet(f"{PASTA}/receitas_tudogostoso.parquet", index=False)
base.drop(columns=["ingredientes", "passos"]).assign(
    ingredientes_texto=lambda d: d["ingredientes_texto"].apply(json.dumps, ensure_ascii=False),
    ingredientes_nomes=lambda d: d["ingredientes_nomes"].apply(json.dumps, ensure_ascii=False),
    passos_texto=lambda d: d["passos_texto"].apply(json.dumps, ensure_ascii=False),
).to_csv(f"{PASTA}/receitas_tudogostoso.csv", index=False)

print(f"✅ Base final: {len(base):,} receitas")
print(f"   com nota: {base['nota'].notna().sum():,} · com tempo: {base['tempo_min'].notna().sum():,} · "
      f"com calorias: {base['calorias'].notna().sum():,}")
print(f"   arquivos: {PASTA}/receitas_tudogostoso.parquet e .csv")
print("\nCategorias mais comuns:"); print(base["categoria"].value_counts().head(10))
print("\nIngredientes mais comuns:")
print(pd.Series([i for l in base["ingredientes_nomes"] for i in l]).value_counts().head(20))